In [1]:
import dlib
import cv2

# Load detektor wajah dari dlib
detector = dlib.get_frontal_face_detector()

# Baca gambar
image = cv2.imread('smile.jpg')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Deteksi wajah
faces = detector(gray)

# Gambar kotak di sekitar wajah
for face in faces:
    x, y, w, h = face.left(), face.top(), face.width(), face.height()
    cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)

cv2.namedWindow("image")
cv2.imshow("image", image)
cv2.waitKey(0)
cv2.destroyWindow('image')
cv2.waitKey(1)

2025-03-10 15:52:23.246 Python[8908:96587] +[IMKClient subclass]: chose IMKClient_Modern
2025-03-10 15:52:23.246 Python[8908:96587] +[IMKInputSession subclass]: chose IMKInputSession_Modern


-1

In [1]:
import cv2
import dlib

# Inisialisasi kamera
cap = cv2.VideoCapture(0)

# Load model deteksi wajah dari dlib (HOG-based detector)
detector = dlib.get_frontal_face_detector()

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠ Gagal menangkap frame!")
        break

    # Konversi ke grayscale untuk deteksi wajah
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Deteksi wajah menggunakan dlib
    faces = detector(gray)

    # Gambar kotak hijau pada wajah yang terdeteksi
    for face in faces:
        x, y, w, h = face.left(), face.top(), face.width(), face.height()
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Tampilkan video dengan deteksi wajah
    cv2.imshow('Face Detection (dlib)', frame)

    # Tekan 'q' untuk keluar dari loop
    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        break

# Lepaskan kamera dan tutup semua jendela
cap.release()


2025-12-01 15:44:51.869 python[13009:195191] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
2025-12-01 15:44:54.163 python[13009:195191] +[IMKClient subclass]: chose IMKClient_Modern
2025-12-01 15:44:54.163 python[13009:195191] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [3]:
import dlib
import cv2
import numpy as np

# Inisialisasi detektor wajah dan model encoding wajah dari dlib
detector = dlib.get_frontal_face_detector()
sp = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
face_rec_model = dlib.face_recognition_model_v1("dlib_face_recognition_resnet_model_v1.dat")

def get_face_encoding(image):
    """Mendeteksi wajah dan mengembalikan encoding wajah."""
    faces = detector(image)
    if len(faces) == 0:
        return None  # Tidak ada wajah terdeteksi

    shape = sp(image, faces[0])
    encoding = np.array(face_rec_model.compute_face_descriptor(image, shape))
    return encoding

# Load gambar referensi dan gambar target
image_ref = cv2.imread("smile.jpg")
image_target = cv2.imread("smile2.jpg")

# Konversi ke grayscale untuk deteksi lebih optimal
image_ref_rgb = cv2.cvtColor(image_ref, cv2.COLOR_BGR2RGB)
image_target_rgb = cv2.cvtColor(image_target, cv2.COLOR_BGR2RGB)

# Ekstrak encoding wajah
encoding_ref = get_face_encoding(image_ref_rgb)
encoding_target = get_face_encoding(image_target_rgb)

# Periksa apakah wajah berhasil dideteksi di kedua gambar
if encoding_ref is None or encoding_target is None:
    print("⚠ Wajah tidak terdeteksi di salah satu gambar!")
else:
    # Bandingkan wajah dengan menghitung jarak Euclidean
    distance = np.linalg.norm(encoding_ref - encoding_target)

    # Threshold standar untuk mengenali wajah
    threshold = 0.6  
    if distance < threshold:
        print("✅ Wajah cocok!")
    else:
        print("❌ Wajah tidak cocok!")

    print(f"🔍 Jarak kesamaan wajah: {distance:.4f}")


✅ Wajah cocok!
🔍 Jarak kesamaan wajah: 0.2454


In [3]:
import dlib
import cv2
import numpy as np

# Load model deteksi wajah dan encoding wajah dari Dlib
detector = dlib.get_frontal_face_detector()
sp = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
face_rec_model = dlib.face_recognition_model_v1("dlib_face_recognition_resnet_model_v1.dat")

def get_face_encoding(image):
    """Mendeteksi wajah dan mengembalikan encoding wajah."""
    faces = detector(image)
    if len(faces) == 0:
        return None  # Tidak ada wajah terdeteksi

    shape = sp(image, faces[0])
    encoding = np.array(face_rec_model.compute_face_descriptor(image, shape))
    return encoding

# Load gambar wajah yang dikenal
image_known = cv2.imread("agung.jpeg")
image_known_rgb = cv2.cvtColor(image_known, cv2.COLOR_BGR2RGB)

# Ambil encoding wajah yang dikenal
known_encoding = get_face_encoding(image_known_rgb)
known_encodings = [known_encoding]
known_names = ["agung suniantara"]

# Inisialisasi webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠ Gagal menangkap frame!")
        break

    # Konversi ke RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Deteksi wajah dalam frame
    faces = detector(rgb_frame)

    for face in faces:
        shape = sp(rgb_frame, face)
        face_encoding = np.array(face_rec_model.compute_face_descriptor(rgb_frame, shape))

        # Bandingkan dengan wajah yang dikenal
        name = "Unknown"
        face_distances = [np.linalg.norm(face_encoding - known_enc) for known_enc in known_encodings]
        best_match_index = np.argmin(face_distances)

        if face_distances[best_match_index] < 0.4:
            name = known_names[best_match_index]

        # Gambar kotak di sekitar wajah
        x, y, w, h = (face.left(), face.top(), face.width(), face.height())
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Tampilkan nama di bawah kotak
        cv2.putText(frame, name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Tampilkan frame
    cv2.imshow("Face Recognition (Dlib)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        break

# Lepaskan kamera dan tutup jendela
cap.release()

In [2]:
import dlib
import cv2
import numpy as np
import os

# Load model deteksi wajah dan encoding wajah dari Dlib
detector = dlib.get_frontal_face_detector()
sp = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
face_rec_model = dlib.face_recognition_model_v1("dlib_face_recognition_resnet_model_v1.dat")

# Fungsi untuk mengambil encoding wajah
def get_face_encoding(image):
    faces = detector(image)
    if len(faces) == 0:
        return None

    shape = sp(image, faces[0])
    encoding = np.array(face_rec_model.compute_face_descriptor(image, shape))
    return encoding

# Fungsi untuk melakukan augmentasi gambar
def augment_image(image):
    augmented_images = [image]
    # Flip horizontal
    augmented_images.append(cv2.flip(image, 1))
    # Brightness adjustment
    for alpha in [0.8, 1.2]:
        bright_image = cv2.convertScaleAbs(image, alpha=alpha, beta=0)
        augmented_images.append(bright_image)
    # Rotate
    for angle in [-15, 15]:
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(image, matrix, (w, h))
        augmented_images.append(rotated)

    return augmented_images

# Load gambar wajah yang dikenal
def load_known_faces(directory):
    known_encodings = []
    known_names = []

    for filename in os.listdir(directory):
        if filename.endswith(".jpeg") or filename.endswith(".jpg"):
            image_path = os.path.join(directory, filename)
            image = cv2.imread(image_path)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Augmentasi gambar
            augmented_images = augment_image(image_rgb)

            for aug_image in augmented_images:
                encoding = get_face_encoding(aug_image)
                if encoding is not None:
                    known_encodings.append(encoding)
                    known_names.append(filename.split(".")[0])

    return known_encodings, known_names

# Load semua gambar di folder 'known_faces'
known_encodings, known_names = load_known_faces("known_faces")

# Inisialisasi webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠ Gagal menangkap frame!")
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    faces = detector(rgb_frame)

    for face in faces:
        shape = sp(rgb_frame, face)
        face_encoding = np.array(face_rec_model.compute_face_descriptor(rgb_frame, shape))

        # Bandingkan dengan wajah yang dikenal
        name = "Unknown"
        face_distances = [np.linalg.norm(face_encoding - known_enc) for known_enc in known_encodings]
        best_match_index = np.argmin(face_distances)

        # Dynamic threshold
        if face_distances[best_match_index] < 0.5:
            name = known_names[best_match_index]

        x, y, w, h = (face.left(), face.top(), face.width(), face.height())
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("Face Recognition (Dlib)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FileNotFoundError: [Errno 2] No such file or directory: 'known_faces'